In [2]:
from google_play_scraper import Sort, reviews
import pandas as pd
from datetime import datetime
import os

def rescrape_im3_with_version():
    APP_ID = 'com.pure.indosat.care'
    T0_BOUNDARY = '2026-04-20'
    OUTPUT_FILE = 'dataset_indosat/myim3_reviews_raw.csv' # Menimpa file lama
    
    print(f"[*] Menginisialisasi ekstraksi ulang untuk: {APP_ID}")
    target_date = datetime.strptime(T0_BOUNDARY, '%Y-%m-%d')
    
    all_reviews = []
    continuation_token = None
    reached_boundary = False
    
    while not reached_boundary:
        result, continuation_token = reviews(
            APP_ID,
            lang='id', 
            country='id', 
            sort=Sort.NEWEST,
            count=1000,
            continuation_token=continuation_token
        )
        
        if not result:
            print("[!] API Google Play berhenti mengembalikan data.")
            break
            
        oldest_in_batch = result[-1]['at']
        all_reviews.extend(result)
        
        print(f" -> Menarik {len(all_reviews)} ulasan... Waktu ulasan tertua di batch: {oldest_in_batch.date()}")
        
        if oldest_in_batch < target_date:
            reached_boundary = True
            print("[*] Batas T0 tercapai.")
            
        if not continuation_token:
            break

    # Konversi ke DataFrame
    df = pd.DataFrame(all_reviews)
    
    # [KUNCI ARSITEKTUR] Menyertakan reviewCreatedVersion secara eksplisit
    target_columns = ['reviewId', 'userName', 'content', 'score', 'at', 'replyContent', 'reviewCreatedVersion']
    
    # Filter hanya kolom yang eksis dari API untuk menghindari KeyError
    available_columns = [col for col in target_columns if col in df.columns]
    df_filtered = df[available_columns].copy()
    
    # Pemangkasan presisi berdasarkan T0
    df_filtered['at'] = pd.to_datetime(df_filtered['at'])
    df_final = df_filtered[df_filtered['at'] >= target_date]
    
    # Ekspor dan timpa CSV mentah sebelumnya
    os.makedirs(os.path.dirname(OUTPUT_FILE), exist_ok=True)
    df_final.to_csv(OUTPUT_FILE, index=False)
    
    print(f"\n[SUCCESS] Pipeline Ingestion Diperbarui.")
    print(f" -> Total Data Tersimpan: {len(df_final)} baris.")
    print(f" -> Kolom yang diekstrak: {list(df_final.columns)}")
    print(f" -> Data diekspor ke: {OUTPUT_FILE}")

# Eksekusi fungsi di cell notebook
rescrape_im3_with_version()

[*] Menginisialisasi ekstraksi ulang untuk: com.pure.indosat.care
 -> Menarik 1000 ulasan... Waktu ulasan tertua di batch: 2026-06-24
 -> Menarik 2000 ulasan... Waktu ulasan tertua di batch: 2026-06-20
 -> Menarik 3000 ulasan... Waktu ulasan tertua di batch: 2026-06-16
 -> Menarik 4000 ulasan... Waktu ulasan tertua di batch: 2026-06-11
 -> Menarik 5000 ulasan... Waktu ulasan tertua di batch: 2026-06-07
 -> Menarik 6000 ulasan... Waktu ulasan tertua di batch: 2026-06-03
 -> Menarik 7000 ulasan... Waktu ulasan tertua di batch: 2026-05-22
 -> Menarik 8000 ulasan... Waktu ulasan tertua di batch: 2026-05-15
 -> Menarik 9000 ulasan... Waktu ulasan tertua di batch: 2026-05-10
 -> Menarik 10000 ulasan... Waktu ulasan tertua di batch: 2026-05-03
 -> Menarik 11000 ulasan... Waktu ulasan tertua di batch: 2026-04-23
 -> Menarik 12000 ulasan... Waktu ulasan tertua di batch: 2026-04-17
[*] Batas T0 tercapai.

[SUCCESS] Pipeline Ingestion Diperbarui.
 -> Total Data Tersimpan: 11566 baris.
 -> Kolom y

In [3]:
import pandas as pd

def check_time_boundary(csv_file):
    print(f"[*] Menginspeksi rentang waktu: {csv_file}")
    try:
        # Kita baca file mentah untuk kecepatan, hanya mengambil kolom 'at'
        df = pd.read_csv(csv_file, usecols=['at'], parse_dates=['at'])
        
        oldest_date = df['at'].min().strftime('%Y-%m-%d %H:%M:%S')
        newest_date = df['at'].max().strftime('%Y-%m-%d %H:%M:%S')
        total_days = (df['at'].max() - df['at'].min()).days
        
        print(f" -> Total Data: {len(df)} baris")
        print(f" -> Tanggal Terlama (T0): {oldest_date}")
        print(f" -> Tanggal Terbaru (T1): {newest_date}")
        print(f" -> Rentang Waktu (Delta): {total_days} hari")
        
    except Exception as e:
        print(f"[ERROR] Inspeksi gagal: {e}")

if __name__ == "__main__":
    check_time_boundary('dataset_indosat/myim3_reviews_raw.csv')

[*] Menginspeksi rentang waktu: dataset_indosat/myim3_reviews_raw.csv
 -> Total Data: 11566 baris
 -> Tanggal Terlama (T0): 2026-04-20 00:15:06
 -> Tanggal Terbaru (T1): 2026-06-28 21:46:02
 -> Rentang Waktu (Delta): 69 hari


In [5]:
def scrap_telkomsel_multi_app():
    # 1. Konfigurasi Target Multi-App
    apps = {
        'Telkomsel_Reguler': 'com.telkomsel.telkomselcm',
        'Telkomsel_Basic': 'com.tsel.telkomselku'
    }
    
    target_dir = 'dataset_telkomsel'
    t0_boundary = pd.to_datetime('2026-04-20')
    
    if not os.path.exists(target_dir):
        os.makedirs(target_dir)
        print(f"[*] Direktori '{target_dir}' berhasil dibuat.")
        
    print(f"[*] Batas bawah waktu analisis (T0): {t0_boundary.strftime('%Y-%m-%d')}")
    
    # Iterasi untuk setiap aplikasi
    for app_name, app_id in apps.items():
        print(f"\n==============================================")
        print(f"[*] Memulai ekstraksi untuk: {app_name} ({app_id})")
        print(f"==============================================")
        
        output_file = os.path.join(target_dir, f'{app_name.lower()}_reviews_raw.csv')
        all_reviews = []
        continuation_token = None
        stop_scraping = False
        batch_counter = 1
        
        while not stop_scraping:
            print(f"    -> Menarik Batch {batch_counter}...")
            
            try:
                result, continuation_token = reviews(
                    app_id,
                    lang='id',
                    country='id',
                    sort=Sort.NEWEST,
                    count=200,
                    continuation_token=continuation_token
                )
                
                if not result:
                    print(f"    [!] Payload kosong untuk {app_name}. Selesai.")
                    break
                    
                for review in result:
                    review_date = pd.to_datetime(review['at'])
                    
                    if review_date < t0_boundary:
                        stop_scraping = True
                        break
                        
                    all_reviews.append({
                        'reviewId': review['reviewId'],
                        'userName': review['userName'],
                        'content': review['content'],
                        'score': review['score'],
                        'thumbsUpCount': review['thumbsUpCount'],
                        'at': review['at'],
                        'reviewCreatedVersion': review['reviewCreatedVersion']
                    })
                    
                if continuation_token is None:
                    break
                    
                batch_counter += 1
                
            except Exception as e:
                print(f"    [ERROR] Terjadi kegagalan API pada {app_name}: {e}")
                break

        # Ekspor ke CSV per Aplikasi
        df = pd.DataFrame(all_reviews)
        if not df.empty:
            df['date'] = pd.to_datetime(df['at']).dt.date
            df = df[df['date'] != df['date'].min()] # Hapus hari parsial
            df.drop(columns=['date'], inplace=True)
            
            df.to_csv(output_file, index=False)
            print(f"    [SUCCESS] {app_name} selesai. Total: {len(df)} keluhan.")
            print(f"    -> Tersimpan di: {output_file}")
        else:
            print(f"    [!] Tidak ada data baru yang ditarik untuk {app_name}.")

if __name__ == "__main__":
    scrap_telkomsel_multi_app()

[*] Batas bawah waktu analisis (T0): 2026-04-20

[*] Memulai ekstraksi untuk: Telkomsel_Reguler (com.telkomsel.telkomselcm)
    -> Menarik Batch 1...
    -> Menarik Batch 2...
    -> Menarik Batch 3...
    -> Menarik Batch 4...
    -> Menarik Batch 5...
    -> Menarik Batch 6...
    -> Menarik Batch 7...
    -> Menarik Batch 8...
    -> Menarik Batch 9...
    -> Menarik Batch 10...
    -> Menarik Batch 11...
    -> Menarik Batch 12...
    -> Menarik Batch 13...
    -> Menarik Batch 14...
    -> Menarik Batch 15...
    -> Menarik Batch 16...
    -> Menarik Batch 17...
    -> Menarik Batch 18...
    -> Menarik Batch 19...
    -> Menarik Batch 20...
    -> Menarik Batch 21...
    -> Menarik Batch 22...
    -> Menarik Batch 23...
    -> Menarik Batch 24...
    -> Menarik Batch 25...
    -> Menarik Batch 26...
    -> Menarik Batch 27...
    -> Menarik Batch 28...
    -> Menarik Batch 29...
    -> Menarik Batch 30...
    -> Menarik Batch 31...
    -> Menarik Batch 32...
    -> Menarik Batch 

In [7]:
from importlib import machinery
import pandas as pd

def check_time_boundary(csv_file):
    print(f"[*] Menginspeksi rentang waktu: {csv_file}")
    try:
        # Kita baca file mentah untuk kecepatan, hanya mengambil kolom 'at'
        df = pd.read_csv(csv_file, usecols=['at'], parse_dates=['at'])
        
        oldest_date = df['at'].min().strftime('%Y-%m-%d %H:%M:%S')
        newest_date = df['at'].max().strftime('%Y-%m-%d %H:%M:%S')
        total_days = (df['at'].max() - df['at'].min()).days
        
        print(f" -> Total Data: {len(df)} baris")
        print(f" -> Tanggal Terlama (T0): {oldest_date}")
        print(f" -> Tanggal Terbaru (T1): {newest_date}")
        print(f" -> Rentang Waktu (Delta): {total_days} hari")
        
    except Exception as e:
        print(f"[ERROR] Inspeksi gagal: {e}")

if __name__ == "__main__":
    check_time_boundary('dataset_telkomsel/telkomsel_basic_reviews_raw.csv')
    check_time_boundary('dataset_telkomsel/telkomsel_reguler_reviews_raw.csv')

[*] Menginspeksi rentang waktu: dataset_telkomsel/telkomsel_basic_reviews_raw.csv
 -> Total Data: 452 baris
 -> Tanggal Terlama (T0): 2026-04-21 01:29:43
 -> Tanggal Terbaru (T1): 2026-06-29 09:47:18
 -> Rentang Waktu (Delta): 69 hari
[*] Menginspeksi rentang waktu: dataset_telkomsel/telkomsel_reguler_reviews_raw.csv
 -> Total Data: 24569 baris
 -> Tanggal Terlama (T0): 2026-04-21 00:02:18
 -> Tanggal Terbaru (T1): 2026-06-29 11:48:40
 -> Rentang Waktu (Delta): 69 hari
